In [1]:
import hashlib

import requests

from api_helper import NorenApiPy
import logging

logging.basicConfig(level=logging.DEBUG)


class FlatData:

    def __init__(self) -> None:
        self.APIKEY = '8ee6ee6a7e8e49e38266639d15bbbad3'
        self.secretKey = '2024.137102b0d10c4750bd9b85471b64c32cce958efce1038018'       
        self.request_code = '' # get this from the url
        self.token = ''
        self.password = '24@Hacker'
        self.userid = 'FT042478'
        self.api = NorenApiPy()
        # self.api.__username = self.userid

    def __sha256(self, text):
        text_bytes = text.encode('utf-8')
        sha256_hash = hashlib.sha256(text_bytes).hexdigest()
        return sha256_hash

    def get_token(self):
        u = 'https://authapi.flattrade.in/trade/apitoken'
        text = self.APIKEY + self.request_code + self.secretKey
        print(text)
        pay = {
            "api_key": self.APIKEY, "request_code": self.request_code, "api_secret": self.__sha256(text)
        }
        r = requests.post(u, json=pay)
        print('Re token',r.text)
        try:
            if r.json()['stat'] == 'ok':
                self.token = r.json()['token']
                return self.token
            else:
                raise ('Token is not available')
        except Exception as e:
            print(e)

    def session(self):

        self.session_req = self.api.set_session(userid=self.userid, password='', usertoken=self.token)

        return self

    def search(self, exchange='NFO', searchtext='46500 CE'):
        result = self.api.searchscrip(exchange=exchange, searchtext=searchtext)
        return result

    def start_socket(self):
        self.api.start_websocket()
        return self
    
    def place_order(self):
        ret = self.api.place_order(buy_or_sell='B', product_type='C',
                        exchange='NSE', tradingsymbol='CANBK-EQ', 
                        quantity=1, discloseqty=0,price_type='SL-LMT', 
                        price=200.00, trigger_price=199.50,
                        retention='DAY', remarks='my_order_001')
        
        self.api.place_order(buy_or_sell='B', product_type='B',
                        exchange='NSE', tradingsymbol='INFY-EQ', 
                        quantity=1, discloseqty=0,price_type='LMT', price=1500, trigger_price=None,
                        retention='DAY', remarks='my_order_001', bookloss_price = 1490, bookprofit_price = 1510)
        
        self.api.place_order(buy_or_sell='B', product_type='C',
                        exchange='NSE', tradingsymbol='INFY-EQ', 
                        quantity=1, discloseqty=0,price_type='MKT', price=0, trigger_price=None,
                        retention='DAY', remarks='my_order_001')


    def sl_modify(self,orderno):
        ret = self.api.modify_order(exchange='NSE', tradingsymbol='CANBK-EQ', orderno=orderno,
                                   newquantity=2, newprice_type='SL-LMT', price_type='MKT',price=0, newprice=201.00, newtrigger_price=200.00)

    def get_positions(self):
        ret = self.api.get_positions()
        mtm = 0
        pnl = 0
        for i in ret:
            mtm += float(i['urmtom'])
            pnl += float(i['rpnl'])
            day_m2m = mtm + pnl
        print(f'{day_m2m} is your Daily MTM')
        return ret

a = FlatData()
print('CALL THIS UL AND LOGIN TO GET THE request_code . https://auth.flattrade.in/?app_key=8ee6ee6a7e8e49e38266639d15bbbad3')
print('and get token and set it in token variable')

CALL THIS UL AND LOGIN TO GET THE request_code . https://auth.flattrade.in/?app_key=8ee6ee6a7e8e49e38266639d15bbbad3
and get token and set it in token variable


In [3]:
a.token = 'add6beacd1fe5806a742a3fb6ca0737ba3300ae6a3733d4548553faf46c0d7bc'
a.session()

DEBUG:NorenRestApiPy.NorenApi:FT042478 session set to : add6beacd1fe5806a742a3fb6ca0737ba3300ae6a3733d4548553faf46c0d7bc


In [ ]:
a.search(searchtext='BANKNIFTY')

In [ ]:
a.api.place_order(buy_or_sell='B', product_type='B',
                        exchange='NFO', tradingsymbol='BANKNIFTY06NOV24C50200', 
                        quantity=15, discloseqty=0,price_type='MKT', price=124.0, trigger_price=None,
                        retention='DAY', remarks='my_order_001', act_id=a.userid)

In [ ]:
# ret = a.api.place_order(buy_or_sell='B', product_type='M',
#                         exchange='NFO', tradingsymbol='BANKNIFTY06NOV24C50200', 
#                         quantity=15, discloseqty=0,price_type='LTD', price=52.4, trigger_price=None,
#                         retention='DAY', remarks='my_order_001',act_id=a.userid)
ret = a.api.place_order(buy_or_sell='B', product_type='M',
                        exchange='NFO', tradingsymbol='BANKNIFTY06NOV24C52900', 
                        quantity=15, discloseqty=0,price_type='MKT', trigger_price=None,
                        retention='DAY', remarks='my_order_001',act_id=a.userid)

In [ ]:
a.api.get_order_book()[0]

In [ ]:
a.api.get_positions()

In [ ]:
q = a.api.get_quotes('NFO', 'BANKNIFTY06NOV24C50200')

In [ ]:
import json
json.loads(json.dumps(q, indent=4))

In [ ]:
data = a.api.get_time_price_series('NFO', 'BANKNIFTY06NOV24P50400', interval='1')

In [ ]:
import pandas as pd
df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
a.api.get_watch_list(wlname='1')

In [ ]:
feed_opened = False

def event_handler_feed_update(tick_data):
    print(f"feed update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True


a.api.start_websocket( order_update_callback=event_handler_feed_update,
                     subscribe_callback=event_handler_feed_update, 
                     socket_open_callback=open_callback, 
                     )

while(feed_opened==False):
    pass

# subscribe to a single token 
a.api.subscribe('NSE|13')

#subscribe to multiple tokens
a.api.subscribe(['NSE|22', 'BSE|522032'])

In [ ]:
data = a.api.get_time_price_series('NFO', 'BANKNIFTY06NOV24P50400', interval='1')

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Sample DataFrame with a 'close' column
# data = {
#     'close': [100, 105, 102, 108, 115, 120, 118]
# }

df = pd.DataFrame(data)

df['close'] = df[['intc']].astype(float)
# Calculate the difference between consecutive 'close' prices
df['close_diff'] = df['close'].diff()

# Calculate the angle in radians, then convert it to degrees
df['angle_degrees'] = np.degrees(np.arctan(df['close_diff']))
df = df.sort_values(by='ssboe', ascending=True)
# Drop the first row with NaN values from diff calculation if needed
df.dropna(inplace=True)

# Create a Plotly figure
fig = go.Figure()

# Add 'close' price line
fig.add_trace(go.Scatter(
    x=df.index, 
    y=df['close'], 
    mode='lines+markers', 
    name='Close Price',
    line=dict(color='blue')
))

# Add 'angle_degrees' line
fig.add_trace(go.Scatter(
    x=df.index, 
    y=df['angle_degrees'], 
    mode='lines+markers', 
    name='Angle Degrees',
    line=dict(color='orange')
))

# Add vertical lines to represent each angle degree
for i in df.index:
    fig.add_shape(
        type="line",
        x0=i, y0=0, x1=i, y1=df['angle_degrees'].loc[i],
        line=dict(color="orange", dash="dot"),
    )

# Customize layout
fig.update_layout(
    title='Close Prices and Angle of Trend',
    xaxis_title='Index',
    yaxis_title='Values',
    showlegend=True,
)

# Add a horizontal line at y=0 for reference
fig.add_shape(
    type="line",
    x0=df.index.min(), y0=0, x1=df.index.max(), y1=0,
    line=dict(color="gray", width=0.5, dash="dash")
)

# Show the plot
fig.show()

In [ ]:
df.sort_values(by='ssboe', ascending=True)

In [ ]:
# Create an empty list to store each row as a dictionary
results = []

# Iterate through the DataFrame and calculate the angle for each close price change
for i in range(1, len(df)):
    close_diff = df['close'].iloc[i] - df['close'].iloc[i - 1]
    angle_degrees = np.degrees(np.arctan(close_diff))
    results.append({'index': i, 'close': df['close'].iloc[i], 'angle_degrees': angle_degrees})

# Convert the list of dictionaries to a DataFrame
result_df = pd.DataFrame(results)

# Plot the results using Plotly
fig = go.Figure()

# Add 'close' price line
fig.add_trace(go.Scatter(
    x=result_df['index'], 
    y=result_df['close'], 
    mode='lines+markers', 
    name='Close Price',
    line=dict(color='blue')
))

# Add 'angle_degrees' line
fig.add_trace(go.Scatter(
    x=result_df['index'], 
    y=result_df['angle_degrees'], 
    mode='lines+markers', 
    name='Angle Degrees',
    line=dict(color='orange')
))

# Add vertical lines to represent each angle degree
for i in result_df['index']:
    fig.add_shape(
        type="line",
        x0=i, y0=0, x1=i, y1=result_df[result_df['index'] == i]['angle_degrees'].values[0],
        line=dict(color="orange", dash="dot"),
    )

# Customize layout
fig.update_layout(
    title='Close Prices and Angle of Trend',
    xaxis_title='Index',
    yaxis_title='Values',
    showlegend=True,
)

# Add a horizontal line at y=0 for reference
fig.add_shape(
    type="line",
    x0=result_df['index'].min(), y0=0, x1=result_df['index'].max(), y1=0,
    line=dict(color="gray", width=0.5, dash="dash")
)

# Show the plot
fig.show()

In [ ]:
import pandas as pd
data = a.api.get_time_price_series('NFO', 'BANKNIFTY06NOV24P50400', interval='1')
df = pd.DataFrame(data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Your data
da = np.array([12, 15, 16, 14, 15, 12, 11, 15, 14, 16])

# Generate x-values for plotting
x = np.arange(len(da))

# Fit linear regression line
z = np.polyfit(x, da, 1)
p = np.poly1d(z)

# Calculate angle in degrees
angle = np.arctan(z[0]) * 180 / np.pi

print(f"Angle of the line: {angle:.2f} degrees")

# Plot data and regression line
plt.plot(x, da, 'o', label='Data')
plt.plot(x, p(x), label='Regression line')
plt.legend()

In [ ]:
df

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import linregress
import math
df['intc'] = df[['intc']].astype(float)
for i, d in df.iterrows():
    # print(i, d['intc'])
    # Data
    da = [df['intc'].iloc[i]]
    print(da)
    x = np.arange(len(da))

    # Perform linear regression
    slope, intercept, _, _, _ = linregress(x, da)

    # Calculate the angle in degrees
    angle = math.degrees(math.atan(slope))

    # Generate points for the regression line
    y_fit = slope * x + intercept

    # Create the plot with Plotly
    fig = go.Figure()

    # Add scatter plot for data points
    fig.add_trace(go.Scatter(x=x, y=da, mode='markers', name='Data Points'))

    # Add line for the trend line
    fig.add_trace(go.Scatter(x=x, y=y_fit, mode='lines', name=f'Fit Line (Angle = {angle:.2f}°)', line=dict(color='red')))

    # Update layout with labels and title
    fig.update_layout(
        title="Data with Trend Line Angle",
        xaxis_title="Index",
        yaxis_title="Value",
        showlegend=True,
        legend=dict(x=0.7, y=1),
    )

    # Display the plot
    fig.show()

# Show the plot
plt.show()

angle


In [ ]:
df.dtypes